# M1.S3 - Architecture of HPC systems
## From system to rack to node to processor

A modern HPC system is hierarchical:

```text
SYSTEM
  |
  +-- RACKS
       |
       +-- NODES
            |
            +-- CPU / GPU
                 |
                 +-- CORES
                 +-- MEMORY
```

The important question is not only **what components exist**, but also **how they are connected** and **where computation and data live**.

### What you will practice

By the end of the notebook you should be able to:

1. read the hierarchy of an HPC system;
2. distinguish a node, socket, core, thread and accelerator;
3. distinguish shared memory, NUMA and distributed memory;
4. explain why an interconnect is essential;
5. classify parallel architectures using Flynn's taxonomy;
6. identify the role of storage and system software;
7. explain why a fast component does not guarantee a fast system;
8. map the real SciTech cluster using architecture vocabulary.

Use the same cycle throughout:

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

## 1 - Build the hierarchy

Put these components in order from largest to smallest:

- core
- system
- node
- rack
- processor/socket

### Your prediction

Write the order here before revealing the answer.

<details>
<summary><strong>Show explanation</strong></summary>

A useful hierarchy is:

```text
system -> rack -> node -> processor/socket -> core
```

A node may contain more than one CPU socket and may also contain one or more GPUs.

</details>

## 2 - Read the real CPU topology

We can inspect the node that hosts the current Jupyter session.

The goal is not to benchmark it. The goal is to identify:

- sockets;
- cores;
- hardware threads;
- NUMA domains;
- memory visible to the operating system.

In [ ]:
import os
import re
import subprocess
from pathlib import Path

def run_command(command):
    print("$", command)
    p = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print("[stderr]")
        print(p.stderr.strip())
    print("return code:", p.returncode)
    print()

print("Architecture inspection only. No heavy computation is launched.")

In [ ]:
run_command("hostname")
run_command("lscpu | grep -E 'Architecture|CPU\\(s\\)|On-line|Thread|Core|Socket|NUMA|Model name'")

### Predict

Suppose `lscpu` reports:

```text
Socket(s):             2
Core(s) per socket:   64
Thread(s) per core:    2
```

How many logical CPUs can the operating system see?

<details>
<summary><strong>Show explanation</strong></summary>

```text
2 sockets x 64 cores/socket x 2 threads/core = 256 logical CPUs
```

A hardware thread is not the same thing as a physical core.

</details>

## 3 - Node does not mean core

These terms describe different levels.

| Term | Meaning |
|---|---|
| **Core** | execution unit inside a CPU |
| **Hardware thread** | logical execution context exposed by a core |
| **Socket / processor** | physical CPU package |
| **NUMA node** | local memory domain inside a server |
| **Compute node** | complete server connected to the cluster |
| **Cluster** | many nodes connected by a network |

### Quick classification

Which term best fits each description?

1. A complete server with CPUs and RAM.
2. A physical CPU package.
3. One execution engine inside that CPU.
4. A memory domain that is closer to some CPU cores than others.
5. A set of servers connected by a high-speed network.

<details>
<summary><strong>Show explanation</strong></summary>

1. Compute node  
2. Socket / processor  
3. Core  
4. NUMA node  
5. Cluster

</details>

## 4 - Shared memory inside a node

Within one shared-memory node, several CPU cores can access the same address space.

Conceptually:

```text
core 0 ----+
core 1 ----+---- shared memory
core 2 ----+
core 3 ----+
```

This is convenient because threads can communicate through ordinary memory.

But shared memory does **not** mean that all memory accesses cost exactly the same.

## 5 - NUMA: shared memory is not equally close

Modern multi-socket HPC nodes often use **Non-Uniform Memory Access (NUMA)**.

A simplified two-socket system looks like this:

```text
CPU socket 0 ---- local memory 0
      |                  |
      +------ link ------+
      |                  |
CPU socket 1 ---- local memory 1
```

Both sockets can access the whole address space, but local memory is generally faster than remote memory.

### Predict

A thread is running on socket 0.

Which placement is usually better?

- its working data is in memory attached to socket 0;
- its working data is in memory attached to socket 1.

<details>
<summary><strong>Show explanation</strong></summary>

The first placement is usually better because the thread accesses **local NUMA memory**.

This is why thread placement and data placement matter.

</details>

In [ ]:
print("NUMA information from the current node:")
run_command("lscpu | grep -E '^NUMA|NUMA node[0-9]+ CPU'")

if subprocess.run(
    "command -v numactl >/dev/null 2>&1",
    shell=True, executable="/bin/bash"
).returncode == 0:
    run_command("numactl --hardware")
else:
    print("numactl is not installed in this environment.")
    print("lscpu still provides the basic NUMA topology.")

### Observe

Look for:

- number of NUMA nodes;
- which CPUs belong to each NUMA node;
- memory associated with each NUMA node, if `numactl` is available.

Do not confuse a **NUMA node inside one server** with a **compute node in the cluster**. They use the same word "node" in different contexts.

## 6 - Distributed memory: each compute node owns its memory

Now move one level higher.

A cluster contains multiple compute nodes:

```text
NODE A                            NODE B
CPU + local memory               CPU + local memory
       |                                |
       +---------- INTERCONNECT --------+
```

A process on Node A cannot directly load a normal memory address from Node B.

Data must be communicated through the network, typically using message passing such as MPI.

### Predict

What changes when we move from shared memory to distributed memory?

<details>
<summary><strong>Show explanation</strong></summary>

With distributed memory:

- each node contributes additional compute;
- each node contributes additional memory;
- data must be partitioned;
- communication becomes explicit;
- network latency and bandwidth become performance factors.

This architecture scales much further, but communication is no longer implicit.

</details>

## 7 - What does the interconnect do?

The interconnect moves data between nodes.

Two properties matter especially:

**Latency**
- how long one communication takes to begin and complete;
- especially important for many small messages.

**Bandwidth**
- how much data can be transferred per second;
- especially important for large messages.

### Match the workload to the likely network pressure

**A.** Millions of tiny synchronization messages.  
**B.** Exchange 20 GB arrays between neighbouring nodes.

<details>
<summary><strong>Show explanation</strong></summary>

- **A:** latency is especially important.
- **B:** bandwidth is especially important.

Real applications often care about both.

</details>

## 8 - Storage is not memory

HPC systems normally contain several data layers:

```text
registers / cache
       |
      DRAM
       |
 local storage / NVMe
       |
 shared / parallel filesystem
       |
 long-term storage
```

They differ in speed, capacity, persistence and scope.

### Classification

Which layer is normally best for each need?

1. Temporary values used every few CPU instructions.
2. Main working arrays of a running program.
3. Large datasets shared by many nodes.
4. Persistent results that must survive after the job finishes.

<details>
<summary><strong>Show explanation</strong></summary>

1. Registers/cache  
2. DRAM  
3. Shared or parallel filesystem  
4. Persistent storage / filesystem

</details>

## 9 - Heterogeneous nodes

Many modern HPC nodes combine different processor types.

A simplified accelerated node:

```text
CPU ---- PCIe / high-speed link ---- GPU
 |                               |
DRAM                            HBM
```

The CPU and GPU are not interchangeable.

**CPU**
- fewer, more general-purpose cores;
- strong control flow and sequential performance.

**GPU**
- many lightweight execution units;
- high throughput for regular parallel work;
- often paired with high-bandwidth memory.

### Predict

Which workload is more naturally suited to a GPU?

- highly regular matrix operations on millions of elements;
- a small branch-heavy control program with little parallel work.

<details>
<summary><strong>Show explanation</strong></summary>

The large regular matrix workload is usually a better GPU candidate because it exposes massive data parallelism.

</details>

## 10 - Flynn's taxonomy

Flynn's taxonomy classifies architectures by the number of instruction and data streams.

| Class | Instruction streams | Data streams | Typical interpretation |
|---|---:|---:|---|
| SISD | 1 | 1 | sequential processor |
| SIMD | 1 | many | same operation on many data elements |
| MISD | many | 1 | uncommon / specialized |
| MIMD | many | many | modern multicore and cluster systems |

### Classify these examples

1. One scalar instruction stream processing one value at a time.
2. One vector instruction applied to eight values.
3. Four independent MPI ranks executing different instructions on different data.
4. A modern CPU+GPU cluster as a whole.

<details>
<summary><strong>Show explanation</strong></summary>

1. SISD  
2. SIMD  
3. MIMD  
4. Primarily MIMD at system level, with SIMD/SIMT-style parallelism inside processors and GPUs.

Modern systems combine multiple forms of parallelism.

</details>

## 11 - System software makes the hardware usable

Users normally do not control hardware directly.

A useful software stack is:

```text
applications
     |
libraries / runtimes
     |
MPI / OpenMP / CUDA / other programming models
     |
scheduler and resource manager
     |
operating system and drivers
     |
CPU / GPU / memory / network / storage
```

### Predict

Where does Slurm belong?

<details>
<summary><strong>Show explanation</strong></summary>

Slurm belongs in the **resource management / scheduler** layer.

It decides which resources a job receives and where that job runs.

</details>

In [ ]:
print("Software stack visible from this environment:")
for cmd in [
    "python3 --version",
    "gcc --version | head -n 1",
    "command -v mpicc || true",
    "command -v srun || true",
    "command -v sbatch || true",
]:
    run_command(cmd)

print("Module system:")
run_command("bash -lc 'type module >/dev/null 2>&1 && echo MODULE_COMMAND=AVAILABLE || echo MODULE_COMMAND=NOT_AVAILABLE'")

## 12 - Inspect the SciTech cluster as a system

We now map the teaching cluster using the same architecture vocabulary.

We will inspect:

- nodes;
- partitions;
- CPUs;
- memory;
- GPU resources;
- current Slurm allocation.

No compute-heavy job is submitted.

In [ ]:
if subprocess.run(
    "command -v sinfo >/dev/null 2>&1",
    shell=True, executable="/bin/bash"
).returncode == 0:
    run_command("sinfo -a -N -o '%N %P %t %c %m %G'")
else:
    print("Slurm commands are not available in this environment.")

In [ ]:
print("Current Jupyter allocation:")
for key in [
    "SLURM_JOB_ID",
    "SLURM_JOB_PARTITION",
    "SLURM_NODELIST",
    "SLURM_CPUS_PER_TASK",
    "SLURM_MEM_PER_CPU",
    "SLURM_MEM_PER_NODE",
]:
    print(f"{key}={os.environ.get(key, 'not set')}")

### Architecture detective

From the actual output, fill in:

1. **Compute nodes:** ...
2. **CPU partition:** ...
3. **GPU partition:** ...
4. **CPUs visible per node:** ...
5. **Memory visible per node:** ...
6. **GPU resources advertised by Slurm:** ...
7. **Current Jupyter allocation:** ...

### Important

The resources visible on a node are not necessarily the resources allocated to your current job.

For example:

```text
node has: 512 logical CPUs
your job has: 2 CPUs
```

You must respect the Slurm allocation.

## 13 - Architecture is a balance

A supercomputer is not only its processors.

A useful system view is:

```text
COMPUTE
   |
MEMORY
   |
INTERCONNECT
   |
STORAGE
   |
SOFTWARE
   |
POWER + COOLING
```

Improving one component can simply move the bottleneck somewhere else.

### Scenario

A system doubles GPU compute capability, but:

- memory bandwidth is unchanged;
- interconnect bandwidth is unchanged;
- storage bandwidth is unchanged.

Will every application become 2x faster?

<details>
<summary><strong>Show explanation</strong></summary>

No.

Only applications limited mainly by GPU computation can approach that improvement.

Memory-bound, communication-bound or I/O-bound applications may improve very little.

A fast component does not make a fast system. **Balance does.**

</details>

## 14 - Architecture detective: JUPITER

Consider this simplified description of JUPITER's accelerated Booster system:

- 5,884 accelerated nodes;
- each node contains 4 NVIDIA GH200 Grace Hopper Superchips;
- each GH200 combines a 72-core Grace CPU and a Hopper GPU;
- CPU and GPU are connected through NVLink-C2C;
- CPU memory: 120 GB LPDDR5X per GH200;
- GPU memory: 96 GB HBM3 per GH200;
- nodes communicate through high-speed InfiniBand NDR;
- the network uses a Dragonfly+ topology.

### Map the architecture

Identify:

1. **Compute**
2. **Accelerator**
3. **CPU memory**
4. **GPU memory**
5. **Node interconnect**
6. **System-level interconnect**
7. **Why the system is heterogeneous**

<details>
<summary><strong>Show explanation</strong></summary>

1. Grace CPUs provide general-purpose compute.  
2. Hopper GPUs provide accelerator compute.  
3. LPDDR5X is attached to the CPU side.  
4. HBM3 is attached to the GPU side.  
5. NVLink-C2C tightly connects CPU and GPU inside each GH200.  
6. InfiniBand NDR connects nodes at system scale.  
7. The system combines different processor and memory technologies optimized for different kinds of work.

</details>

## 15 - What architecture would you choose?

Choose the resources you would prioritize.

### A - Large CFD simulation
Many communicating processes and large numerical computation.

### B - Train a large neural network
Highly parallel matrix operations and large datasets.

### C - Analyze a 3 TB graph in memory
Irregular access and a very large memory requirement.

### D - Thousands of independent simulations
Little communication between jobs.

Available resource ideas:

- CPU nodes;
- GPU nodes;
- large-memory nodes;
- fast interconnect;
- local NVMe.

<details>
<summary><strong>Show explanation</strong></summary>

A reasonable mapping:

- **A - CFD:** CPU/GPU compute plus a very strong interconnect.
- **B - AI:** GPUs, HBM and fast GPU/network communication.
- **C - Graph:** large-memory node; CPU may be appropriate for irregular access.
- **D - Independent simulations:** many CPU or GPU nodes; interconnect performance is less critical.

There is no universally best HPC architecture.

The architecture should match the workload's **computation, memory and communication pattern**.

</details>

## 16 - Mini-practice: map a real HPC system

Use the SciTech cluster output from this notebook.

Create a diagram using no more than **six boxes**.

A possible structure is:

```text
SciTech cluster
      |
  CPU nodes
      |
 CPUs + DRAM
      |
interconnect
      |
 GPU resources
      |
Slurm / software
```

Your version should reflect the actual evidence you observed.

### Then answer

> **What workloads does this architecture seem suitable for, and why?**

Use architecture vocabulary:

- compute;
- memory;
- accelerator;
- interconnect;
- storage;
- shared/distributed memory;
- heterogeneous.

## 17 - Exit ticket

Answer each in one or two sentences.

### 1. What is the difference between a core and a compute node?

Your answer:

### 2. What does NUMA mean?

Your answer:

### 3. Why does distributed memory require communication?

Your answer:

### 4. What does MIMD describe?

Your answer:

### 5. Why is system balance important?

Your answer:

<details>
<summary><strong>Show explanation</strong></summary>

- **Core vs node:** a core is one execution unit inside a processor; a compute node is a complete server.
- **NUMA:** one shared address space where memory access cost depends on where the data is physically located.
- **Distributed memory:** one node cannot directly load ordinary memory belonging to another node, so data must be exchanged explicitly.
- **MIMD:** multiple instruction streams operate on multiple data streams.
- **Balance:** application performance depends on compute, memory, communication, storage and software working together.

</details>

## What you should leave with

- HPC systems are **hierarchical**: system -> rack -> node -> processor -> core.
- Memory is **shared inside a node** but may be non-uniform because of NUMA.
- Memory is **distributed between nodes**, so communication is explicit.
- The interconnect determines how effectively nodes can cooperate.
- Modern systems are often **heterogeneous**, combining CPUs, GPUs and different memory technologies.
- Flynn's taxonomy helps describe forms of parallel architecture.
- Storage and software are part of the architecture, not afterthoughts.
- A fast component does not make a fast system. **Balance does.**
- Architecture should match the workload.

Next: **M1.S4 - Resource management and job scheduling**.